# SynAgent: generate and rank synthesis routes

This notebook does two things with a molecule you want to make:

1. Generate: you give SynAgent a target molecule and your constraints, and it proposes several synthesis routes. Every step comes from its libraries of reaction templates and purchasable building blocks, and SynAgent checks that each reaction gives the product it claims.
2. Rank: SynAgent orders those routes by how feasible they are to carry out, taking your own priorities into account.

SynAgent runs on Google's Gemini model in this notebook. To use it, you need a free Google account.

## One-time setup

You only need to do these six steps once. If someone has already set SynAgent up on your computer, skip to "Opening the notebook" below.

### 1. Open a terminal

A terminal is a window where you type commands. Paste each command below into it and press Enter.

- macOS: press `⌘ Space`, type `Terminal`, and press Enter.
- Windows: open the Start menu, type `PowerShell`, and press Enter.
- Linux: open your Terminal application.

### 2. Install git

git downloads SynAgent and one of the packages it depends on.

- macOS: type `git --version` and press Enter. If git isn't installed, macOS offers to install it. Click **Install** and wait for it to finish.
- Windows: download the installer from https://git-scm.com/downloads/win and run it with the default options.
- Linux: install it with your package manager, for example `sudo apt install git`.

### 3. Install uv

uv installs Python and the packages SynAgent needs into the project's own folder, so it doesn't change anything else on your computer.

macOS or Linux:

```sh
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Windows (PowerShell):

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Then close the terminal and open a new one so it can find `uv`.

### 4. Download and install SynAgent

```sh
git clone https://github.com/lukasmki/SynAgent.git
cd SynAgent
uv sync --extra jupyter
```

The first command creates a `SynAgent` folder, the second moves into it, and the third installs everything. The third one takes a few minutes the first time and prints a lot of text, which is normal.

### 5. Get a Gemini API key

An API key works like a password that lets SynAgent use Gemini on your behalf.

1. Go to https://aistudio.google.com/api-keys and sign in with a Google account.
2. Click **Create API key**. If it asks for a project, accept the default.
3. Copy the key. It's a long string that starts with `AIza`.

Don't share the key or paste it into this notebook. Google's free tier has usage limits, and the same page shows how much you've used.

### 6. Save the key in a `.env` file

SynAgent reads the key from a file named `.env` in the `SynAgent` folder. In the terminal, still inside that folder, run the command for your system. Replace `paste-your-key-here` with the key you copied, and keep the single quotes around it.

macOS or Linux:

```sh
echo 'GOOGLE_API_KEY=paste-your-key-here' > .env
```

Windows (PowerShell):

```powershell
Set-Content -Path .env -Value 'GOOGLE_API_KEY=paste-your-key-here'
```

The file now holds one line, `GOOGLE_API_KEY=AIza...`. Finder and File Explorer hide files whose names start with a dot, so you may not see it. The project's git settings ignore the file, so it stays on your computer and is never uploaded to GitHub.

## Opening the notebook

Do this each time you want to use the notebook. Open a terminal and run:

```sh
cd SynAgent
uv run --extra jupyter jupyter lab notebook-examples/generate-and-rank.ipynb
```

The notebook opens in your web browser. Keep the terminal window open while you work, because closing it stops the notebook.

### Running the cells

- To run a code cell, click it and press **Shift + Enter**. Run the cells in order from top to bottom.
- A running cell shows `[*]` to its left. The `*` turns into a number when the cell finishes.
- Generating routes takes a few minutes, and so does ranking them. SynAgent shows its progress while it works.
- If you change a setting, run that cell again, then the cells below it.
- To start over, choose **Kernel → Restart Kernel** from the menu and run everything again from Step 1.

## Step 1: Check your setup

Click the cell below and press **Shift + Enter**. It should say that it found your API key. If it doesn't, see [If something goes wrong](#If-something-goes-wrong) at the bottom of the notebook.

In [ ]:
import os

import synagent as sa
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if os.environ.get("GOOGLE_API_KEY"):
    print("✅ Found your Gemini API key. You're ready to go.")
else:
    print(
        "❌ No Gemini API key found.\n\n"
        "Check steps 5 and 6 of the setup above: the SynAgent folder needs a file\n"
        "named .env containing one line, GOOGLE_API_KEY=your-key. Once you've fixed\n"
        "it, use Kernel → Restart Kernel in the menu, then run this cell again."
    )

## Step 2: Start SynAgent

This cell starts SynAgent with Gemini. The model runs on Google's servers, so you don't need a GPU.

The cell prints the configuration, the model, and the groups of tools SynAgent can use. With them it searches for building blocks and reaction templates, checks that each reaction gives the product it should, and looks up prices if you have a Chemspace key.

You can leave `MODEL` as it is. To try a different Gemini model, change the name after `google:`.

In [ ]:
MODEL = "google:gemini-3.8-flash"

agent = sa.SynAgent(spec="nogpu", model=MODEL)
agent

## Step 3: Generate routes

### Edit this

- `TARGET` is the molecule you want to make, written as a SMILES string. To find one, look the molecule up on [PubChem](https://pubchem.ncbi.nlm.nih.gov/) and copy the SMILES under "Names and Identifiers". You can also draw it in ChemDraw or [Ketcher](https://lifescience.opensource.epam.com/KetcherDemoSA/index.html) and copy it as SMILES. Keep the quotes around it.
- `NUMBER_OF_ROUTES` is how many different routes to propose. Start with three; asking for more takes longer.
- `CONSTRAINTS` lists anything the routes must respect, in plain English, one per line. Keep the triple quotes (`"""`) at the start and end. If you have no constraints, write `None` between them.

The example target is an amide loosely related to imatinib. Replace it with your own.

Some other constraints you could use: *at most 4 steps*, *avoid protecting groups*, *avoid cryogenic conditions*, *start from commercially available anilines*, *keep the piperazine intact*.

In [ ]:
TARGET = "Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1"

NUMBER_OF_ROUTES = 3

CONSTRAINTS = """
At most 3 steps.
Avoid palladium-catalysed couplings where there is a reasonable alternative.
"""

Run the cell below to generate the routes. It takes a few minutes.

While it works, SynAgent prints each tool it calls in a yellow box, such as a structure check or a building block search, and what the tool returned in a green box. You can ignore these. The routes are written up at the end.

In [ ]:
chat = agent.chat()

routes = await chat.ask(f"""
Propose {NUMBER_OF_ROUTES} different synthesis routes for the target {TARGET}.

Constraints from the user:
{CONSTRAINTS}

Validate every route before presenting it. Present each one under a heading
"Route N", with its steps in forward order. For each step give the reaction name,
the reactant and product SMILES, and whether it passed validation. Then list the
route's building blocks. Say clearly if a route could not be fully validated, or
if a constraint could not be met.
""")

## Step 4: Rank the routes

SynAgent now ranks the routes it just proposed. It always judges synthetic feasibility: whether every step passed validation, how many steps a route has, how reliable each reaction is in practice, and whether the building blocks are available. It then weighs your own priorities.

### Edit this

`RANKING_PRIORITIES` lists what matters most to you, in plain English, one per line. Keep the triple quotes (`"""`) at the start and end. To rank on feasibility alone, write `None` between them.

You could also ask it to *prefer routes that avoid chromatography*, *prefer routes I can run in a day*, or *prefer building blocks I can buy from one supplier*.

In [ ]:
RANKING_PRIORITIES = """
Prefer routes with the fewest steps.
Prefer common, inexpensive building blocks.
Penalise steps that need careful handling or specialised equipment.
"""

In [ ]:
ranking = await chat.ask(f"""
Rank the routes you just proposed, best first.

Judge synthetic feasibility on:
- Validation: a route with failed or unresolved steps ranks below every verified route.
- Length: fewer steps is better, and a convergent route beats a linear one of the same length.
- Reliability: how robust each reaction is in practice, and how likely side reactions,
  competing reactive groups, or protecting groups are to cause trouble.
- Building blocks: whether each one is in the library and how common it is.

Then weigh these priorities from the user:
{RANKING_PRIORITIES}

Present the ranking as a numbered list, best first. For each route give one line on
why it sits there, its number of steps, and its main risk. Where a route ranks lower
only because of the user's priorities rather than its feasibility, say so. Finish with
one short paragraph recommending a route and what to check before trying it at the bench.
""")

## Step 5: Save the results

This cell saves the target, your settings, the routes, and the ranking to a Markdown file named `synagent-results.md`. The file goes in the `notebook-examples` folder, and the cell prints its full path. Running the cell again overwrites the file, so change `OUTPUT_FILE` first if you want to keep an earlier set of results.

In [ ]:
from datetime import datetime
from pathlib import Path

OUTPUT_FILE = "synagent-results.md"

Path(OUTPUT_FILE).write_text(
    f"# SynAgent results\n\n"
    f"Target: `{TARGET}`  \n"
    f"Model: {MODEL}  \n"
    f"Date: {datetime.now():%Y-%m-%d %H:%M}\n\n"
    f"## Constraints\n{CONSTRAINTS}\n\n"
    f"## Routes\n\n{routes.text}\n\n"
    f"## Ranking priorities\n{RANKING_PRIORITIES}\n\n"
    f"## Ranking\n\n{ranking.text}\n"
)

print(f"✅ Saved to {Path(OUTPUT_FILE).resolve()}")

## Step 6: Ask follow-up questions (optional)

SynAgent remembers this whole conversation, so you can ask more about the routes it proposed. Change the question in `FOLLOW_UP` and run the cell, as many times as you like.

Some questions to try:

- *Why did you rank the top route above the others?*
- *Are there alternative building blocks for step 2 of Route 1?*
- *Propose one more route that avoids the piperazine alkylation.*
- *What could go wrong with the amide coupling in Route 2?*

Running the Step 3 cell again starts a new conversation.

In [ ]:
FOLLOW_UP = "Why did you rank the top route above the others?"

await chat.ask(FOLLOW_UP)

## If something goes wrong

### "No Gemini API key found"

The `.env` file is missing, in the wrong folder, or has a typo. It belongs in the `SynAgent` folder next to `README.md`, and it should contain one line: `GOOGLE_API_KEY=your-key`. After you fix it, choose **Kernel → Restart Kernel** and run the cells again from the top.

### A message about a quota, a rate limit, or `429`

You've reached the free Gemini tier's limits. Wait a few minutes and run the cell again, or check your usage at https://aistudio.google.com/api-keys.

### "ModuleNotFoundError: No module named 'synagent'"

Jupyter was started outside the project's environment. Close it, then start it again from inside the `SynAgent` folder with `uv run --extra jupyter jupyter lab notebook-examples/generate-and-rank.ipynb`.

### A cell seems stuck

Generating and ranking take a few minutes, and new tool boxes keep appearing while SynAgent works. If nothing new appears for a long time, choose **Kernel → Interrupt Kernel** and run the cell again.

### SynAgent can't validate a route

When SynAgent can't ground a step in its libraries, it reports the step as unresolved. Try a different target, loosen your constraints, or use the follow-up cell to ask why a step failed.

### Prices and availability

SynAgent looks up prices on Chemspace, which needs its own API key. Without one, SynAgent tells you that pricing is unavailable. To turn it on, add a second line to `.env`: `CHEMSPACE_API_KEY=your-key`.

## Where to go next

- `notebook-examples/notebook.ipynb` covers the rest of the Python interface, including tool call inspection, routes built by hand, and the validation workflows.
- The documentation explains the command line tool, the other configurations, and how to change what SynAgent can do. Build it with `make -C docs html`.